# Reconstructing a grating from its own diffracted waves
A periodic amplitude mask can reproduce its structure in empty space. Change the period while holding wavelength fixed. Decide whether the recurrence distance grows linearly or quadratically with period.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

period_um = 12.
wavelength_um = .6
phase_rad = .55
assert 0<wavelength_um/period_um<.45

## Propagate each Fourier order
Choose a smooth mask with two pairs of nonzero diffraction orders, avoiding ambiguous aperture-edge sampling. The longitudinal wave number uses the decaying square-root branch. Removing the common carrier leaves every intensity unchanged.

In [ ]:
n=256
u=np.arange(n)/n
mask=.58+.22*np.cos(2*np.pi*u)+.12*np.cos(4*np.pi*u+phase_rad)
m=np.fft.fftfreq(n)*n
spectrum=np.fft.fft(mask)
k=2*np.pi/wavelength_um; kx=2*np.pi*m/period_um
kz=np.sqrt(k*k-kx*kx+0j); ztalbot=2*period_um**2/wavelength_um
def field(frac,exact=False):
    phase=(kz-k)*ztalbot if exact else -2*np.pi*m*m
    return np.fft.ifft(spectrum*np.exp(1j*np.asarray(frac)[...,None]*phase),axis=-1)
print(f"Paraxial field recurrence distance {ztalbot:.4f} μm.")
print("RMS intensity discrepancy after one paraxial length:",
      np.sqrt(np.mean((abs(field(1,True))**2-mask**2)**2)))

## Read the carpet before studying the cuts
Dark and bright structures need not track geometrical ray paths. The image on the left uses exact angular-spectrum phases; compare the same grating at a quarter, half and full paraxial Talbot distance on the right.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
z=np.linspace(0,1,240);image=abs(field(z,True))**2
im=ax[0].pcolormesh(u,z,image,shading="auto",cmap="viridis")
fig.colorbar(im,ax=ax[0],label="Intensity / incident intensity")
ax[0].set(xlabel="Lateral position x/d",ylabel="Propagation distance z/zT",title="Exact propagation of the mask")
for f,c,style in [(0.,"#222222","-"),(.25,"#375b9a",":"),(.50,"#175868","-."),(1.,"#a23d21","--")]:
    ax[1].plot(u,abs(field(f,True))**2,color=c,ls=style,label=f"z/zT = {f:g}")
ax[1].set(xlabel="Lateral position x/d",ylabel="Relative intensity",title="Planes of the Talbot carpet");ax[1].legend(fontsize=8)
plt.show()

## Verify phase, translation and energy
A half-recurrence is a translation by half a period for any periodic input within the paraxial model. At a quarter-recurrence the odd Fourier orders are multiplied by minus i. Exact propagation converges towards these predictions at smaller wavelength-to-period ratio. The chosen mask contains only propagating orders.

In [ ]:
assert np.allclose(field(1.),mask)
assert np.allclose(field(.5),np.roll(mask,n//2))
quarter=(1-1j)/2*mask+(1+1j)/2*np.roll(mask,n//2)
assert np.allclose(field(.25),quarter)
assert np.isclose(np.mean(abs(field(1.,True))**2),np.mean(mask**2))

## Optional OneSimulation angular-spectrum solve
This comparison exercises the library's forward or decaying longitudinal branch and its propagation operator with the same physical field. The Fourier-order derivation above remains independent.

In [ ]:
try:
    from onesim.physics.optics.propagation.numerics.angular_spectrum import AngularSpectrumPropagation, longitudinal_wavenumber
except ImportError:
    print("The optional optical propagation check runs locally with OneSimulation.")
else:
    longitudinal=longitudinal_wavenumber(kx[:,None],k)
    propagator=AngularSpectrumPropagation(longitudinal=longitudinal,forward=np.fft.fft,inverse=np.fft.ifft,reference_wavenumber=k)
    result=propagator(mask,ztalbot)
    print("Maximum complex-field discrepancy:",np.max(abs(result-field(1.,True))))
    assert np.allclose(result,field(1.,True),atol=1e-11)